# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library, focusing on record and field access via their `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields and their `@id` values.

We'll print all record sets with their name and `@id`, then for each record set, print its fields with their `@id`, label, and data type.

In [ ]:
# Get record set objects
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, label: {field.label}, data type: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: extract data from all record sets
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for RecordSet: {record_set.name} (@id={record_set.id})")
    records = list(dataset.records(record_set=record_set.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set.id] = df
        print(f"Columns in DataFrame: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set.name} (@id={record_set.id})")
    print("")
# For further analysis, select a main tabular record set
# As an example, select the first non-empty one
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id is not None:
    print(f"Main record set ID chosen for EDA: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
else:
    print('No tabular record sets found with records.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

> **Note:** Replace the values below with the appropriate field `@id`s found in Section 2 above to match available numeric/categorical fields in the chosen record set.

In [ ]:
if main_record_set_id is not None:
    df = main_df.copy()
    # You may need to adjust these field ids to fit your dataset
    # Use Section 2 printout for correct field `@id`

    # Example field names -- update as appropriate!
    # Let's try to find a suitable numeric field. We'll guess based on typical proteomics fields.
    possible_numeric_fields = [c for c in df.columns if ('coverage' in c.lower() or 'count' in c.lower() or 'abundance' in c.lower() or 'mw' in c.lower() or 'weight' in c.lower())]

    print(f"Numeric-like fields: {possible_numeric_fields}")
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]

    # We'll use 10 as a threshold for demonstration. You can adjust based on the actual field value distributions.
    threshold = 10
    numeric_field_id = numeric_field  # The actual column name (@id)
    print(f"Filtering records where '{numeric_field_id}' > {threshold}...")
    
    # Convert to numeric, coerce errors
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by possible categorical fields (e.g. 'description', 'sample', 'accession')
    possible_group_fields = [c for c in df.columns if ('desc' in c.lower() or 'sample' in c.lower() or 'type' in c.lower() or 'accession' in c.lower())]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print('No obvious group/categorical field found.')
else:
    print('No main DataFrame to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
if main_record_set_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If group field exists, show group means as a barplot
    if group_field:
        plt.figure(figsize=(10, 5))
        group_means = filtered_df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=group_means.index, y=group_means.values, palette='viridis')
        plt.xticks(rotation=45, ha='right')
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook showed how to load, explore, and analyze the FAIR<sup>2</sup> mass spectrometry extracellular vesicle dataset using `mlcroissant` and field `@id`s.
- Field and record set inspection allows precise referencing and reproducibility.
- Initial EDA and visualizations reveal the distribution and grouping of key numeric variables (e.g., abundance, coverage, counts).
- For further biological insights, join with external annotation tables or consult the Data Dictionary in the metadata.